# Notebook 02 — Logistic Regression
## Predicting Yes or No

**Dataset**: Titanic passengers
**Question we answer**: *Did this passenger survive?*
**Learning type**: Supervised Binary Classification — the answer is 0 or 1

---

## Section 1 — What Is Logistic Regression?

### In plain English

Linear regression predicts a number. But sometimes the answer is just **yes or no**.

**Logistic regression wraps linear regression in an S-shaped filter called the sigmoid:**

```
linear score = w1×pclass + w2×sex + w3×age + ... + bias    ← any number
probability  = 1 / (1 + e^(−score))                        ← always 0.0 to 1.0
```

The probability is then thresholded at 50%:
- 0.85 → 85% chance survived → **predict: Survived**
- 0.30 → 30% chance survived → **predict: Did Not Survive**

> **Why not just use linear regression for yes/no?**
> Linear regression can output −3 or 2.7 — meaningless as probabilities.
> Sigmoid squashes any number into [0, 1], making it interpretable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-7, 7, 300)
sig = 1 / (1 + np.exp(-x))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, sig, 'steelblue', lw=2.5)
ax.axhline(0.5, color='green', ls='--', lw=1.5, label='Decision boundary (50%)')
ax.fill_between(x, 0.5, sig, where=(sig > 0.5), alpha=0.12, color='green', label='Predict: Survived')
ax.fill_between(x, sig, 0.5, where=(sig < 0.5), alpha=0.12, color='red',   label='Predict: Not survived')
ax.set_xlabel('Linear score (wx + b)', fontsize=12)
ax.set_ylabel('Probability of survival', fontsize=12)
ax.set_title('Sigmoid: Converts Any Number to a Probability', fontsize=12)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
print('No matter how large or small the linear score, sigmoid always outputs 0–1.')

## Section 2 — How Does It Learn?

### Same mechanics as linear regression — different error measure

**Step 1** — Compute linear score → sigmoid → probability
**Step 2** — Measure error with **Log-Loss** (cross-entropy):
```
loss = −[ y × log(p) + (1−y) × log(1−p) ]
```
- Passenger survived (y=1), model predicted p=0.90 → small loss ✓
- Passenger survived (y=1), model predicted p=0.10 → huge loss ✗

**Step 3** — Gradient descent: adjust weights to reduce log-loss. Repeat.

| | Linear Regression | Logistic Regression |
|---|---|---|
| Output | Any number | Probability 0–1 |
| Error function | Mean Squared Error | Log-Loss |
| Learning method | Gradient descent | Gradient descent |
| Needs scaling | Yes | **Yes (same reason)** |

## Section 3 — What Does the Data Need?

Logistic regression uses gradient descent — **same scaling requirements as linear regression**.

| Feature | Distribution | Transform | Why |
|---|---|---|---|
| `pclass` | Ordinal 1–3 | StandardScaler | Uncentred |
| `sex_enc` | Binary | StandardScaler | Applied uniformly |
| `age` | ≈ Normal | StandardScaler | Centre and scale |
| `sibsp` | Zero-heavy | StandardScaler | Sparse range |
| `parch` | Zero-heavy | StandardScaler | Sparse range |
| `log_fare` | ≈ Normal after log | StandardScaler | Log first, then centre |
| `embarked_enc` | Ordinal | StandardScaler | Applied uniformly |
| `family_size` | Skewed count | StandardScaler | Wide range |

> We use `log_fare` (not raw fare) because raw fare is heavily right-skewed.
> Log transform makes it roughly normal before scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

df_raw = sns.load_dataset('titanic')

df = df_raw.copy()
df['age']          = df['age'].fillna(df['age'].median())
df['fare']         = df['fare'].fillna(df['fare'].median())
df['embarked']     = df['embarked'].fillna(df['embarked'].mode()[0])
df['sex_enc']      = (df['sex'] == 'female').astype(int)
df['embarked_enc'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df['family_size']  = df['sibsp'] + df['parch'] + 1
df['log_fare']     = np.log1p(df['fare'])
df = df.dropna(subset=['survived'])

FEATURES = ['pclass','sex_enc','age','sibsp','parch','log_fare','embarked_enc','family_size']
TARGET   = 'survived'

X_raw = df[FEATURES].values
y     = df[TARGET].values.astype(int)

X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_raw)
X_test  = scaler.transform(X_te_raw)

print(f"Train: {len(X_train)} passengers  ({y_train.mean()*100:.1f}% survived)")
print(f"Test : {len(X_test)}  passengers  ({y_test.mean()*100:.1f}% survived)")

## Section 4 — Train the Model and Read the Rules

The weights in logistic regression are **log-odds coefficients**:
- Positive weight → feature increases the odds of survival
- Negative weight → feature decreases the odds
- Larger |weight| → stronger effect on the prediction

For example, a large positive weight on `sex_enc` (female=1) means being female
strongly increases predicted survival probability — matching historical reality.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc*100:.1f}%  ({int(acc*len(y_test))}/{len(y_test)} passengers correct)")
print()
print(classification_report(y_test, y_pred, target_names=['Did not survive','Survived']))

coef_df = pd.DataFrame({'Feature': FEATURES, 'Weight': model.coef_[0].round(3)})
coef_df = coef_df.reindex(coef_df['Weight'].abs().sort_values(ascending=False).index)
print("Feature weights (sorted by magnitude):")
print(coef_df.to_string(index=False))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Not survived','Survived'],
    cmap='Blues', ax=ax1, colorbar=False)
ax1.set_title(f'Confusion Matrix\nAccuracy = {acc*100:.1f}%', fontsize=12)

colors = ['#26A69A' if w >= 0 else '#EF5350' for w in coef_df['Weight']]
ax2.barh(coef_df['Feature'], coef_df['Weight'], color=colors)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_title('Feature Weights\n(green = increases survival odds  red = decreases)', fontsize=12)
ax2.set_xlabel('Weight (log-odds coefficient)', fontsize=11)

plt.tight_layout()
plt.show()
print()
print('sex_enc has the largest positive weight → being female strongly raises survival odds')
print('pclass  has a negative weight           → higher class number = lower class = lower odds')